# GovSpend Capstone - Normalization Pipeline and Results

DS6015 / UVA MSDS / Sponsor: GovSpend

This notebook runs the full quantity-normalization pipeline end to end and evaluates whether it works. It does not require a language model to run, so the results here stand on their own.

The question it answers: if we convert purchase order line items to a per-each price, does price variability actually go down, and do the outliers we would flag change?

Sections 1 to 4 build the pipeline. Sections 5 to 8 test it. Section 9 prints a summary block to paste into chat or into the report. Section 10 is the optional language model stage, which is off by default.

## 0. Setup

In [1]:
import re
import json
import numpy as np
import pandas as pd

In [2]:
# Merge the raw GovSpend exports into one file.
# This only needs to run once. If the merged file is already on disk we skip it, so the
# notebook still runs for anyone who has the merged extract but not the raw parts.
from pathlib import Path

MERGED_PATH = Path("export_merged_with_uom_and_category.csv")
RAW_PARTS   = [Path("export_part1.csv"), Path("export_part2.csv"), Path("export_part3.csv")]

REBUILD_MERGED = not MERGED_PATH.exists()

if not REBUILD_MERGED:
    print(f"{MERGED_PATH} already exists, skipping the merge and rebuild steps.")
    print("Delete that file, or set REBUILD_MERGED = True by hand, to rebuild from the raw parts.")
elif all(p.exists() for p in RAW_PARTS):
    df = pd.concat([pd.read_csv(p, low_memory=False) for p in RAW_PARTS], ignore_index=True)
    print(f"Merged {len(RAW_PARTS)} raw exports into {len(df):,} rows")
else:
    missing = [str(p) for p in RAW_PARTS if not p.exists()]
    raise FileNotFoundError(
        f"Neither {MERGED_PATH} nor the raw parts were found. Missing: {missing}")

export_merged_with_uom_and_category.csv already exists, skipping the merge and rebuild steps.
Delete that file, or set REBUILD_MERGED = True by hand, to rebuild from the raw parts.


In [3]:
# Clean unit of measure and category, then write the merged extract.
# Skipped entirely when the merged file already exists.

if REBUILD_MERGED:
    df["LineItem.UOM"] = (
        df["LineItem.UOM"]
        .astype("string")
        .str.strip()
        .replace(["", "nan", "NaN", "<NA>", "None", "NULL"], pd.NA))

    df["LineItem.UOM"] = df["LineItem.UOM"].replace({
        "Each": "EA", "EACH": "EA", "each": "EA", "ea": "EA",
        "Box": "BX", "BOX": "BX", "box": "BX",
        "Case": "CS", "CASE": "CS", "CSE": "CS"})

    df["LineItem.Category"] = (
        df["LineItem.Category"]
        .astype("string")
        .str.strip()
        .replace(["", "nan", "NaN", "<NA>", "None", "NULL"], pd.NA))

    # keep only rows that have both a unit of measure and a category
    df = df.dropna(subset=["LineItem.UOM", "LineItem.Category"]).copy()
    print(f"Rows after filtering: {len(df):,}")

    df.to_csv(MERGED_PATH, index=False)
    print(f"Created {MERGED_PATH}")
else:
    print(f"Using the existing {MERGED_PATH}")

Using the existing export_merged_with_uom_and_category.csv


In [4]:
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

CSV_PATH = 'export_merged_with_uom_and_category.csv'
SAMPLE_ROWS = None   # set to 100000 for a fast pass

COL_DESC='LineItem.Description'; COL_QTY='LineItem.Quantity'
COL_UPRICE='LineItem.UnitPrice'; COL_UOM='LineItem.UOM'
COL_TOTAL='LineItem.TotalPrice'; COL_CAT='LineItem.Category'

RESULTS = {}   # collects headline numbers for the summary block in section 9

df = pd.read_csv(CSV_PATH, nrows=SAMPLE_ROWS)
RESULTS['rows_loaded'] = len(df)
print(f'Loaded {len(df):,} line items')

Loaded 917,321 line items


## 1. Data integrity check

Before anything else, confirm the arithmetic is internally consistent. If total equals quantity times unit price everywhere, then the numbers themselves are fine and the only thing blocking comparison is that the unit differs from row to row. That is the premise of the whole project.

In [5]:
chk = df.dropna(subset=[COL_QTY,COL_UPRICE,COL_TOTAL])
identity = ((chk[COL_QTY]*chk[COL_UPRICE] - chk[COL_TOTAL]).abs() < 0.02).mean()
RESULTS['price_identity_pct'] = round(identity*100,2)
print(f'Total = Qty x UnitPrice holds for {identity*100:.2f}% of rows')
print()
print('Missingness (%):')
print((df[[COL_DESC,COL_QTY,COL_UPRICE,COL_UOM,COL_CAT]].isna().mean()*100).round(2).to_string())

Total = Qty x UnitPrice holds for 100.00% of rows

Missingness (%):
LineItem.Description    0.0
LineItem.Quantity       0.0
LineItem.UnitPrice      0.0
LineItem.UOM            0.0
LineItem.Category       0.0


## 2. Unit of measure classification

We sort every unit code into four buckets. Each-like codes are already per item. Service codes are time based (hours, months) and describe services, which cannot be converted to an each quantity at all. Multi-or-measure codes are the cases, boxes and packs that actually need converting. Blank units are dropped.

We also treat some codes as synonyms, because different source systems write the same container differently. CR and CS both mean case, BX and BOX both mean box.

In [6]:
SERVICE_UOMS  = {'HUR','MON','DAY','ANN','WEE','YR','HR','MO','WK','MIN'}
EACHLIKE_UOMS = {'EA','CT','PC','PCS','UN','UNT','ST'}
SYN = {'CS':'CS','CR':'CS','CASE':'CS','CSE':'CS','PK':'PK','PKG':'PK','PACK':'PK',
       'BX':'BX','BO':'BX','BOX':'BX','CT':'CT','CTN':'CT','CARTON':'CT','EA':'EA','EACH':'EA'}
INNER = {'CS':{'PK','BX'}, 'BX':{'PK'}}

def canon(u):
    return SYN.get(str(u).strip().upper(), str(u).strip().upper())

def uom_class(u):
    if pd.isna(u) or str(u).strip()=='': return 'no-uom'
    v = str(u).strip().upper()
    if v in SERVICE_UOMS:  return 'service'
    if v in EACHLIKE_UOMS: return 'each-like'
    return 'multi-or-measure'

df['uom_class'] = df[COL_UOM].map(uom_class)
print('Distinct UOM codes:', df[COL_UOM].nunique())
print(df['uom_class'].value_counts().to_string())
RESULTS['uom_class_counts'] = df['uom_class'].value_counts().to_dict()

Distinct UOM codes: 215
uom_class
each-like           693107
multi-or-measure    202643
service              21571


## 3. Non-item filter

Some rows are not products. They are Excel errors, freight charges, tax lines, or invoice summary rows. These can never have a pack size and would distort any price statistic, so we remove them from the working set.

In [7]:
junk_re = re.compile(r'^\s*(#NAME\?|shipping|freight|tax\b|discount|item$|order summary|misc\b|-?inv#|invoice|lab supplies|parts for)', re.I)
df['is_junk'] = df[COL_DESC].astype(str).str.match(junk_re)
RESULTS['junk_rows'] = int(df['is_junk'].sum())
RESULTS['junk_pct'] = round(df['is_junk'].mean()*100,2)
print(f"Non-item rows removed: {df['is_junk'].sum():,} ({df['is_junk'].mean()*100:.2f}%)")
print(df.loc[df['is_junk'], COL_DESC].value_counts().head(6).to_string())

work   = df[(df['uom_class']!='no-uom') & (~df['is_junk'])].copy()
target = work[work['uom_class']=='multi-or-measure'].copy()
ea_ref = work[work['uom_class']=='each-like'].copy()
RESULTS['working_rows']=len(work); RESULTS['target_rows']=len(target); RESULTS['ea_rows']=len(ea_ref)
print()
print(f'Working set : {len(work):,}')
print(f'  needs conversion : {len(target):,}')
print(f'  already each     : {len(ea_ref):,}')

Non-item rows removed: 38,057 (4.15%)
LineItem.Description
#NAME?                11392
Shipping               4466
Freight                1773
Item                   1478
Order Summary item     1386
shipping               1236

Working set : 879,264
  needs conversion : 198,337
  already each     : 659,617


## 4. Pack size parser

Reads pack quantities that are already written into the description and checks the unit it found against the unit the row is priced in before accepting a value.

The rules, in priority order. A nested token like "10/BX 20BX/CS" on a case row gives 10 times 20, or 200. A direct token like "500/CS" on a case row gives 500. If the only count found belongs to a container smaller than the one being priced, for example a "100/PK" note on a row priced by the case, we refuse to answer, because 100 is items per pack and not items per case, and using it would undercount badly. Weaker phrasings like "case of 24" are accepted at medium confidence.

Every answer carries the rule that produced it and the exact text it matched, so any single conversion can be checked by hand.

In [8]:
_U        = r'(?:CS|CSE|CASE|BX|BOX|PK|PKG|PACK|CT|CTN|CARTON)'
tok_re    = re.compile(r'(\d{1,7})\s*/\s*(' + _U + r'|EA)\b', re.I)
inner_tok = re.compile(r'(\d{1,7})\s*(' + _U + r')\s*/\s*(' + _U + r')\b', re.I)
phrase_re = re.compile(r'\b(?:case|box|pack|pkg|carton|ctn|bag|set|kit|bundle)\s*of\s*(\d{1,5})\b', re.I)
per_re    = re.compile(r'(\d{1,7})\s*per\s+(case|box|pack|carton|cs|bx|pk|ct)\b', re.I)
# the unit has to end the word, otherwise '40X48 LINERS' reads the L of LINERS as litres
xmult_re  = re.compile(r'(?<![\d.])(\d{1,4})\s*[xX]\s*\d{1,4}\s*(?:ML|GAL|OZ|L|G)(?![A-Za-z])', re.I)

# which containers can sit inside which, used to combine two separate counts
NEST = {'CS': ['BX','PK'], 'BX': ['PK'], 'CT': ['BX','PK']}

def parse_pack(desc, uom):
    # strip thousands separators so '1,000/CS' is read as 1000 and not 000
    s = re.sub(r'(?<=\d),(?=\d{3})', '', str(desc))
    u=canon(uom); cls=uom_class(uom)
    if cls=='service':   return (np.nan,'service-not-convertible','n/a','')
    if cls=='each-like': return (1.0,'uom-is-each','high',str(uom))

    levels={}
    for m in tok_re.finditer(s): levels[canon(m.group(2))]=int(m.group(1))
    outer={}
    for m in inner_tok.finditer(s): outer[canon(m.group(3))]=(canon(m.group(2)),int(m.group(1)))

    # form one: '100/BX 10BX/CS' gives inner count and inner-per-outer together
    if u in outer:
        iu, po = outer[u]
        if iu in levels:
            return (float(levels[iu]*po),'nested-token','high',f'{levels[iu]}/{iu} x {po}{iu}/{u}')

    # form two: two separate counts, one for an inner container and one for this unit,
    # e.g. '1000/Pack - 10 / Box' priced by the box is 1000 x 10 and not 10
    if u in levels and u in NEST:
        for iu in NEST[u]:
            if iu in levels:
                return (float(levels[iu]*levels[u]),'nested-direct-pair','high',
                        f'{levels[iu]}/{iu} x {levels[u]}/{u}')

    if u in levels:
        return (float(levels[u]),'direct-token','high',f'{levels[u]}/{u}')
    if u in INNER and any(k in levels for k in INNER[u]):
        return (np.nan,'inner-unit-only','none','')
    if levels:
        return (np.nan,'unit-mismatch','none','')

    m=phrase_re.search(s)
    if m: return (float(m.group(1)),'phrase-of','medium',m.group(0))
    m=per_re.search(s)
    if m:
        ev=canon(m.group(2))
        if ev==u: return (float(m.group(1)),'per-token','medium',m.group(0))
        if u in INNER and ev in INNER[u]: return (np.nan,'inner-unit-only','none','')
        return (np.nan,'unit-mismatch','none','')
    m=xmult_re.search(s)
    if m: return (float(m.group(1)),'x-multiplier','medium',m.group(0))
    return (np.nan,'unresolved','none','')

In [9]:
parsed = target.apply(lambda r: parse_pack(r[COL_DESC], r[COL_UOM]), axis=1, result_type='expand')
parsed.columns = ['pack_size','method','confidence','evidence']
target = pd.concat([target, parsed], axis=1)
# a pack size of zero or less is a parse failure, not an answer
bad_pack = target['pack_size'].notna() & (target['pack_size'] <= 0)
target.loc[bad_pack, ['pack_size','confidence','method']] = [np.nan,'none','invalid-pack-size']
print(f'Rejected {int(bad_pack.sum())} rows with a non-positive parsed pack size')

target['unit_price_each'] = target[COL_UPRICE] / target['pack_size']
target['qty_each']        = target[COL_QTY]    * target['pack_size']

resolved_mask = target['confidence'].isin(['high','medium'])
RESULTS['resolved_rows'] = int(resolved_mask.sum())
RESULTS['resolved_pct']  = round(resolved_mask.mean()*100,1)
RESULTS['method_counts'] = target['method'].value_counts().to_dict()
RESULTS['declined_rows'] = int(target['method'].isin(['inner-unit-only','unit-mismatch']).sum())

print(target['method'].value_counts().to_string())
print()
print(f'Resolved: {resolved_mask.sum():,} of {len(target):,} ({resolved_mask.mean()*100:.1f}%)')
print(f'Deliberately declined (ambiguous unit): {RESULTS["declined_rows"]:,}')
target.loc[resolved_mask, [COL_DESC,COL_UOM,COL_UPRICE,'pack_size','unit_price_each','method','evidence']].head(8)

Rejected 10 rows with a non-positive parsed pack size
method
unresolved            152332
direct-token           38757
nested-token            2284
inner-unit-only         1912
unit-mismatch            953
phrase-of                940
per-token                764
x-multiplier             264
nested-direct-pair       121
invalid-pack-size         10

Resolved: 43,130 of 198,337 (21.7%)
Deliberately declined (ambiguous unit): 2,865


,LineItem.Description,LineItem.UOM,LineItem.UnitPrice,pack_size,unit_price_each,method,evidence
87,Business Source Colored Fold-back Binder Clips - Mini - 0.6' Width...,PK,14.97,100.0,0.149700,direct-token,100/PK
112,BIC Gel-ocity Gel Pen - 0.7 mm Medium Pen Point - Retractable - As...,PK,12.84,8.0,1.605000,direct-token,8/PK
143,Swingline Optima High Capacity Staples - 125 Per Strip - High Capa...,BX,5.85,2500.0,0.002340,direct-token,2500/BX
144,"Pathways Soak Proof Shield Heavyweight Paper Plates, 10.13' dia, G...",PK,31.68,125.0,0.253440,direct-token,125/PK
248,MT10013CV-DMEMW/GLN HIGLU PYR6X500 6/CS,CS,41.38,6.0,6.896667,direct-token,6/CS
249,MT10040CV-RPMI 1640 W/GLN 6X500ML 6/CS,CS,37.73,6.0,6.288333,direct-token,6/CS
366,0540218-TUBE SAFE-LCK 500UL NAT 500/PK,PK,75.80,500.0,0.151600,direct-token,500/PK
440,Post-itreg Super Sticky Notes - 1080 - 3' Flag/Note Width x 3' Fla...,PK,13.22,12.0,1.101667,direct-token,12/PK


## 5. Headline test: does normalization reduce price variability?

This is the test that decides whether the project premise holds. We use coefficient of variation, which is standard deviation divided by mean, so it is comparable across items at different price levels.

We run it at two levels of grouping. First on rows that share an identical item description, which means they really are the same product bought in different pack sizes. Second on the category and unit groupings currently used in the aggregation, which is much coarser. The comparison between these two is the interesting part.

In [10]:
def cv(s):
    s = pd.Series(s).dropna()
    if len(s) < 2 or s.mean() == 0: return np.nan
    return s.std()/s.mean()

r = target[resolved_mask].copy()
r = r[(r[COL_UPRICE] > 0) & (r['unit_price_each'] > 0) & np.isfinite(r['unit_price_each'])]
r['norm'] = r[COL_DESC].astype(str).str.upper().str.strip()

# Level 1: identical description, more than one pack size present
rows=[]
for nm, g in r.groupby('norm'):
    if len(g) < 10 or g['pack_size'].nunique() < 2: continue
    a, b = cv(g[COL_UPRICE]), cv(g['unit_price_each'])
    if pd.notna(a) and pd.notna(b):
        rows.append({'desc':nm[:60],'n':len(g),'packs':g['pack_size'].nunique(),'cv_raw':a,'cv_norm':b,
                     'raw_range':f'{g[COL_UPRICE].min():.2f}-{g[COL_UPRICE].max():.2f}',
                     'each_range':f'{g["unit_price_each"].min():.3f}-{g["unit_price_each"].max():.3f}'})
SAME = pd.DataFrame(rows)
if len(SAME):
    SAME['improved'] = SAME['cv_norm'] < SAME['cv_raw']
    RESULTS['same_item_groups']    = len(SAME)
    RESULTS['same_item_improved']  = int(SAME['improved'].sum())
    RESULTS['same_item_cv_raw']    = round(SAME['cv_raw'].median(),3)
    RESULTS['same_item_cv_norm']   = round(SAME['cv_norm'].median(),3)
    RESULTS['same_item_change_pct']= round(((SAME['cv_norm']-SAME['cv_raw'])/SAME['cv_raw']).median()*100,1)
    print('SAME PRODUCT, DIFFERENT PACK SIZES')
    print(f'  groups tested              : {len(SAME)}')
    print(f'  variability went down in   : {SAME["improved"].sum()} ({SAME["improved"].mean()*100:.0f}%)')
    print(f'  median CV before           : {SAME["cv_raw"].median():.3f}')
    print(f'  median CV after            : {SAME["cv_norm"].median():.3f}')
    print(f'  median change              : {RESULTS["same_item_change_pct"]:+.0f}%')
else:
    print('no qualifying same-product groups in this sample')

SAME PRODUCT, DIFFERENT PACK SIZES
  groups tested              : 58
  variability went down in   : 58 (100%)
  median CV before           : 0.653
  median CV after            : 0.078
  median change              : -91%


In [11]:
# the examples worth showing a sponsor
if len(SAME):
    ex = SAME[SAME['cv_raw'] > 0.3].sort_values('cv_raw', ascending=False).head(8)
    print('Apparent price spread before normalization vs after:')
    print(ex[['desc','n','packs','raw_range','each_range','cv_raw','cv_norm']].to_string(index=False))
    RESULTS['top_examples'] = ex.head(4)[['desc','raw_range','each_range','cv_raw','cv_norm']].to_dict('records')

Apparent price spread before normalization vs after:
                                                       desc  n  packs    raw_range  each_range   cv_raw  cv_norm
    CONTROL, BLD GLUCOSE LEVEL 3 TRUE METRIX (1/BX 24BX/CS) 30      2  6.03-144.81 6.030-6.034 2.377782 0.000114
COVER, PROBE ULTRSND SHEATH LATEX 1-1/4 X8 (100/BX 24BX/CS) 14      2 21.58-529.96 0.216-0.225 2.330938 0.012820
    CONTROL, BLD GLUCOSE LEVEL 1 TRUE METRIX (1/BX 24BX/CS) 27      2  6.03-144.81 6.030-6.034 2.270855 0.000166
                FOAM, ADH SIL W/BORDER 3 X3 (10/BX 20BX/CS) 19      2 25.33-506.60 2.533-2.565 2.176235 0.005277
                 BATTERY, F/HEARING AID SZ13 (8/PK 60PK/CS) 10      2 10.24-614.67 1.280-1.472 1.937767 0.061509
                FOAM, ADH SIL W/BORDER 4 X4 (10/BX 20BX/CS) 23      2 34.93-698.60 3.493-3.497 1.880818 0.000514
                  BATTERY, HEARING AID SZ312 (8/PK 60PK/CS) 17      2  8.42-614.67 1.052-1.472 1.733701 0.058273
                         HAIRBRUSH, ADLT LF

In [12]:
# Level 2: the coarse grouping currently used for aggregation
rows=[]
for (cat,uom), g in r.groupby([COL_CAT, COL_UOM]):
    if len(g) < 30: continue
    a,b = cv(g[COL_UPRICE]), cv(g['unit_price_each'])
    if pd.notna(a) and pd.notna(b):
        rows.append({'cat':str(cat)[:45],'uom':uom,'n':len(g),'items':g[COL_DESC].nunique(),'cv_raw':a,'cv_norm':b})
CAT = pd.DataFrame(rows)
if len(CAT):
    CAT['improved'] = CAT['cv_norm'] < CAT['cv_raw']
    RESULTS['cat_groups']     = len(CAT)
    RESULTS['cat_improved']   = int(CAT['improved'].sum())
    RESULTS['cat_change_pct'] = round(((CAT['cv_norm']-CAT['cv_raw'])/CAT['cv_raw']).median()*100,1)
    RESULTS['cat_median_items_per_group'] = int(CAT['items'].median())
    print('CATEGORY + UNIT GROUPS (the coarse grouping)')
    print(f'  groups tested            : {len(CAT)}')
    print(f'  variability went down in : {CAT["improved"].sum()} ({CAT["improved"].mean()*100:.0f}%)')
    print(f'  median change            : {RESULTS["cat_change_pct"]:+.0f}%')
    print(f'  median distinct products per group: {CAT["items"].median():.0f} across {CAT["n"].median():.0f} rows')

CATEGORY + UNIT GROUPS (the coarse grouping)
  groups tested            : 190
  variability went down in : 31 (16%)
  median change            : +56%
  median distinct products per group: 28 across 75 rows


## 6. Why the two levels disagree

If normalization helps on true product groups but hurts on category groups, the explanation matters. If pack size is unrelated to price inside a group, then dividing by pack size adds a second independent source of variation instead of removing one, and measured variability rises. That would mean the category groups are mixing products that were never comparable to begin with, and the rising variability is a symptom of the grouping, not of the conversion.

This cell tests that directly by measuring how strongly pack size and price move together inside the coarse groups.

In [13]:
cors=[]
for (cat,uom), g in r.groupby([COL_CAT, COL_UOM]):
    if len(g) < 30 or g['pack_size'].nunique() < 3: continue
    gg = g[(g['pack_size']>0) & (g[COL_UPRICE]>0)]
    if len(gg) < 30: continue
    cors.append(np.corrcoef(np.log(gg['pack_size']), np.log(gg[COL_UPRICE]))[0,1])
cors = pd.Series(cors).dropna()
if len(cors):
    RESULTS['packsize_price_corr'] = round(cors.median(),3)
    print(f'Groups tested: {len(cors)}')
    print(f'Median correlation between pack size and unit price: {cors.median():.2f}')
    print('A value near zero means pack size is not what is driving price differences inside these groups,')
    print('which is what we would expect if the groups contain many unrelated products.')

Groups tested: 169
Median correlation between pack size and unit price: 0.06
A value near zero means pack size is not what is driving price differences inside these groups,
which is what we would expect if the groups contain many unrelated products.


## 7. Unit of measure reliability

While validating the synonym mapping we noticed rows coded as each whose descriptions clearly state a case quantity and whose prices look like case prices. This matters because the each rows are the reference we compare everything else against. This cell estimates how often it happens.

In [14]:
pack_tok = r'\d{1,6}\s*/\s*(?:CS|BX|PK|CT|CASE|BOX|PACK|PKG|CTN|CARTON)\b'
ea_ref = ea_ref.copy()
ea_ref['has_pack_token'] = ea_ref[COL_DESC].astype(str).str.contains(pack_tok, case=False, regex=True)
n_sus = int(ea_ref['has_pack_token'].sum())
RESULTS['ea_suspect_rows'] = n_sus
RESULTS['ea_suspect_pct']  = round(ea_ref['has_pack_token'].mean()*100,2)
print(f'Rows coded as each: {len(ea_ref):,}')
ea_ref['has_pack_token'] = ea_ref[COL_DESC].astype(str).str.contains(pack_tok, case=False, regex=True)
print()
print('Examples to review:')
print(ea_ref.loc[ea_ref['has_pack_token'], [COL_DESC, COL_UOM, COL_UPRICE]].head(6).to_string(index=False))

Rows coded as each: 659,617

Examples to review:
                                                                                                                                                                            LineItem.Description LineItem.UOM  LineItem.UnitPrice
                                                                                                WP Pallet-Tite Pallet Wrap - 16' Width x 1500 ft Length x 0.5 mil Thickness - Clear - 4 / Carton           CT               73.09
                         Clorox Disinfecting Cleaning Wipes Value Pack - For Multi Surface - Ready-To-Use - Fresh, Citrus Blend Scent - Pre-moistened, Disposable - White - 35 Tub - 15 / Carton           CT               91.90
                                                                                                                                      Starbucks, SBK12411960, Pike Place Ground Coffee, 18 / Box           EA               27.41
Hammermill Great White Recycled Copy Paper - Wh

## 8. Does this change which purchases get flagged?

The practical question for the sponsor is whether normalization changes the answer, not just the numbers. We flag outliers using a robust score on log prices, which is not thrown off by a few extreme values the way a standard deviation is, and count how many rows change status once quantities are on a common basis.

In [15]:
def robust_z(s):
    med = s.median(); mad = (s - med).abs().median()
    # a near-zero MAD makes the score explode, so require a meaningful spread
    if not np.isfinite(mad) or mad < 1e-6: return s*0
    return (s - med)/(1.4826*mad)

flips = total = 0
for nm, g in r.groupby('norm'):
    if len(g) < 10 or g['pack_size'].nunique() < 2: continue
    o_raw  = robust_z(np.log(g[COL_UPRICE])).abs() > 3.5
    o_norm = robust_z(np.log(g['unit_price_each'])).abs() > 3.5
    flips += int((o_raw != o_norm).sum()); total += len(g)
if total:
    RESULTS['outlier_flip_rows']=flips; RESULTS['outlier_flip_total']=total
    RESULTS['outlier_flip_pct']=round(flips/total*100,1)
    print(f'Rows whose outlier status changed after normalization: {flips:,} of {total:,} ({flips/total*100:.1f}%)')
    print('These are purchases we would have judged differently using the raw unit price.')

Rows whose outlier status changed after normalization: 220 of 1,539 (14.3%)
These are purchases we would have judged differently using the raw unit price.


## 9. Summary block

In [16]:
print('='*64)
print('GOVSPEND NORMALIZATION PIPELINE - RESULTS SUMMARY')
print('='*64)
for k,v in RESULTS.items():
    if k in ('top_examples','method_counts','uom_class_counts'): continue
    print(f'{k:32s} {v}')
print()
print('Method breakdown:', json.dumps(RESULTS.get('method_counts',{})))
print()
print(json.dumps({k:v for k,v in RESULTS.items() if k not in ('top_examples',)}, default=str)[:2000])

target.loc[resolved_mask].to_csv('resolved_baseline.csv', index=False)
target[target['confidence']=='none'][[COL_DESC,COL_UOM,COL_QTY,COL_UPRICE,'method']].to_csv('unresolved_for_llm.csv', index=False)
# Gold set for hand labeling.
# Stratified so there are enough parser-answered rows to measure precision and enough
# declined rows to measure what a model could still recover.
#
# IMPORTANT: once the team fills this in by hand the labels are irreplaceable, so we never
# overwrite an existing gold set. If the file is on disk we leave it exactly as it is.

GOLD_LABEL_PATH = Path('gold_set_to_label.csv')
GOLD_KEY_PATH   = Path('gold_set_ANSWERKEY.csv')

if GOLD_LABEL_PATH.exists():
    _existing = pd.read_csv(GOLD_LABEL_PATH)
    _filled = int(pd.to_numeric(_existing['true_pack_size'], errors='coerce').notna().sum()) \
              if 'true_pack_size' in _existing.columns else 0
    print(f'{GOLD_LABEL_PATH} already exists with {_filled} numeric labels. Not overwriting.')
    print('Delete the file by hand if you deliberately want a fresh sample.')
else:
    res_pool = target.loc[resolved_mask]
    unr_pool = target[target['confidence']=='none']

    def take(pool, n, seed=1):
        return pool.sample(min(n, len(pool)), random_state=seed) if len(pool) else pool

    parts = [
        (take(res_pool[res_pool['confidence']=='high'], 110), 'parser-high'),
        (take(res_pool[res_pool['confidence']=='medium'], 40), 'parser-medium'),
        (take(unr_pool[unr_pool['method']=='inner-unit-only'], 40), 'declined-inner-unit'),
        (take(unr_pool[unr_pool['method']=='unit-mismatch'], 20), 'declined-mismatch'),
        (take(unr_pool[unr_pool['method']=='unresolved'], 90), 'no-token'),
    ]
    cols = [COL_DESC, COL_UOM, COL_QTY, COL_UPRICE, 'pack_size','method','confidence','evidence']
    frames=[]
    for d, bucket in parts:
        if not len(d): continue
        f = d[cols].copy(); f['bucket']=bucket; frames.append(f)
    gold = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=7).reset_index(drop=True)
    gold.insert(0,'row_id', range(1,len(gold)+1))
    gold['true_pack_size']=''; gold['labeler']=''; gold['notes']=''
    gold.to_csv(GOLD_KEY_PATH, index=False)
    gold.drop(columns=['pack_size','method','confidence','evidence']).to_csv(GOLD_LABEL_PATH, index=False)
    print(f'Gold set created: {len(gold)} rows')
    print(gold['bucket'].value_counts().to_string())

print()
print('Wrote resolved_baseline.csv and unresolved_for_llm.csv')

GOVSPEND NORMALIZATION PIPELINE - RESULTS SUMMARY
rows_loaded                      917321
price_identity_pct               100.0
junk_rows                        38057
junk_pct                         4.15
working_rows                     879264
target_rows                      198337
ea_rows                          659617
resolved_rows                    43130
resolved_pct                     21.7
declined_rows                    2865
same_item_groups                 58
same_item_improved               58
same_item_cv_raw                 0.653
same_item_cv_norm                0.078
same_item_change_pct             -91.2
cat_groups                       190
cat_improved                     31
cat_change_pct                   56.0
cat_median_items_per_group       27
packsize_price_corr              0.056
ea_suspect_rows                  9775
ea_suspect_pct                   1.48
outlier_flip_rows                220
outlier_flip_total               1539
outlier_flip_pct                 

## 10. Optional language model stage

Off by default. This is the stage that would attack the rows the parser could not resolve, using retrieved examples of similar items the parser did resolve plus similar items priced individually as a price reference. It is not needed for any result above, and we have not committed to a model yet, so it stays disabled until that decision and the data handling question are settled.

In [17]:
RUN_LLM = False

if RUN_LLM:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    corpus = target.loc[target['confidence']=='high'].copy()
    corpus['norm'] = corpus[COL_DESC].astype(str).str.upper().str.strip()
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=200000)
    mat = vec.fit_transform(corpus['norm'])

    def retrieve(q, k=3, min_sim=0.25):
        s = cosine_similarity(vec.transform([str(q).upper()]), mat).flatten()
        idx = s.argsort()[::-1][:k]
        out = corpus.iloc[idx].copy(); out['similarity']=s[idx]
        return out[out['similarity']>=min_sim]

    print(retrieve('NITRILE EXAM GLOVES LARGE')[[COL_DESC,'pack_size','similarity']])
else:
    print('Language model stage disabled. Nothing above depends on it.')

Language model stage disabled. Nothing above depends on it.


## 11. Learned model for pack size

Everything up to this point is rules and descriptive statistics, not a learned model. This section adds one.

The idea is weak supervision. Our parser has already produced about 43,000 rows where it is confident of the pack size, and it produced them for free. We treat those as training labels and train a classifier to predict pack size from the item description, then apply it to the rows the parser could not resolve. The parser writes the labels, the model learns to generalize beyond the rows that happened to spell the pack size out.

One thing we have to be careful about. If we leave the pack token in the description, the model just reads the answer off the text and learns nothing. So we mask those tokens out before training. We also split the data by product rather than at random, because the same product appears many times in this file and a random split would let the model memorize products in training and score well on the same products in test. The gap between those two evaluations is reported below and is worth understanding.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack

lab = target.loc[resolved_mask & target['pack_size'].notna()].copy()
vc  = lab['pack_size'].value_counts()
lab = lab[lab['pack_size'].isin(vc[vc>=30].index)]
print(f'Training rows from the parser: {len(lab):,} across {lab["pack_size"].nunique()} pack sizes')

# mask the pack token so the model cannot simply read the answer
mask_re = r'\d{1,6}\s*(?:BX|PK)?\s*/\s*(?:CS|CSE|BX|BO|PK|CT|CASE|BOX|PACK|PKG|CTN|CARTON|EA)\b'
lab['masked'] = lab[COL_DESC].fillna('').astype(str).str.replace(mask_re,' ',regex=True)
lab['product'] = lab['masked'].str.upper().str.strip().str[:60]
y = lab['pack_size'].astype(str)

def fit_eval(tr_idx, te_idx, label):
    A = lab.iloc[tr_idx]; B = lab.iloc[te_idx]
    v  = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=120000, min_df=2)
    uv = TfidfVectorizer(analyzer='word')
    Xtr = hstack([v.fit_transform(A['masked']), uv.fit_transform(A[COL_UOM].fillna('').astype(str))]).tocsr()
    Xte = hstack([v.transform(B['masked']),     uv.transform(B[COL_UOM].fillna('').astype(str))]).tocsr()
    m = LogisticRegression(max_iter=1000).fit(Xtr, y.iloc[tr_idx])
    pred = m.predict(Xte); conf = m.predict_proba(Xte).max(axis=1)
    yte = y.iloc[te_idx]
    base = yte.value_counts(normalize=True).iloc[0]*100
    print(f'\n{label}')
    print(f'  majority-class baseline : {base:.1f}%')
    print(f'  overall accuracy        : {accuracy_score(yte,pred)*100:.1f}%')
    rows=[]
    for t in [0.5,0.7,0.9]:
        k = conf>=t
        if k.sum():
            a=accuracy_score(yte[k],pred[k])*100
            print(f'  confidence >= {t}      : covers {k.mean()*100:.0f}% of rows at {a:.1f}% accuracy')
            rows.append({'threshold':t,'coverage_pct':round(k.mean()*100,1),'accuracy_pct':round(a,1)})
    return accuracy_score(yte,pred)*100, rows

tr,te = train_test_split(np.arange(len(lab)), test_size=0.25, random_state=42, stratify=y)
acc_rand, _ = fit_eval(tr,te,'RANDOM SPLIT (optimistic - same product can appear in train and test)')

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr,te = next(gss.split(lab, groups=lab['product']))
acc_grp, grp_rows = fit_eval(tr,te,'PRODUCT SPLIT (honest - every test product is unseen)')

RESULTS['ml_acc_random_split']  = round(acc_rand,1)
RESULTS['ml_acc_product_split'] = round(acc_grp,1)
RESULTS['ml_operating_points']  = grp_rows
print(f'\nGap between the two evaluations: {acc_rand-acc_grp:.1f} points.')
print('That gap is how much of the random-split score came from seeing the same product twice.')

Training rows from the parser: 42,210 across 61 pack sizes

RANDOM SPLIT (optimistic - same product can appear in train and test)
  majority-class baseline : 11.8%
  overall accuracy        : 88.6%
  confidence >= 0.5      : covers 73% of rows at 98.9% accuracy
  confidence >= 0.7      : covers 58% of rows at 99.6% accuracy
  confidence >= 0.9      : covers 31% of rows at 99.9% accuracy

PRODUCT SPLIT (honest - every test product is unseen)
  majority-class baseline : 11.8%
  overall accuracy        : 60.6%
  confidence >= 0.5      : covers 46% of rows at 85.2% accuracy
  confidence >= 0.7      : covers 24% of rows at 96.6% accuracy
  confidence >= 0.9      : covers 11% of rows at 99.9% accuracy

Gap between the two evaluations: 28.0 points.
That gap is how much of the random-split score came from seeing the same product twice.


## 12. What the model is worth in practice

A model is only useful here if it converts rows the parser could not. This applies the product-split model to the unresolved rows and reports how many additional conversions we would get at each confidence level, so we can choose an operating point rather than accept whatever the model outputs.

In [19]:
v  = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=120000, min_df=2)
uv = TfidfVectorizer(analyzer='word')
Xall = hstack([v.fit_transform(lab['masked']), uv.fit_transform(lab[COL_UOM].fillna('').astype(str))]).tocsr()
final = LogisticRegression(max_iter=1000).fit(Xall, y)

unres = target[target['confidence']=='none'].copy()
unres['masked'] = unres[COL_DESC].fillna('').astype(str).str.replace(mask_re,' ',regex=True)
Xu = hstack([v.transform(unres['masked']), uv.transform(unres[COL_UOM].fillna('').astype(str))]).tocsr()
unres['pred_pack'] = final.predict(Xu)
unres['pred_conf'] = final.predict_proba(Xu).max(axis=1)

print(f'Unresolved rows scored: {len(unres):,}')
for t in [0.5,0.7,0.9]:
    k = unres['pred_conf']>=t
    add = int(k.sum())
    tot = RESULTS['resolved_rows'] + add
    print(f'  at confidence >= {t}: {add:,} additional rows, total coverage {tot/RESULTS["target_rows"]*100:.1f}% (from {RESULTS["resolved_pct"]}%)')

RESULTS['ml_added_at_0.7'] = int((unres['pred_conf']>=0.7).sum())
unres.loc[unres['pred_conf']>=0.7, [COL_DESC,COL_UOM,COL_UPRICE,'pred_pack','pred_conf']].to_csv('model_predictions.csv', index=False)
print('\nWrote model_predictions.csv')
print('These still need checking against the hand-labeled gold set before we trust them.')

Unresolved rows scored: 155,207
  at confidence >= 0.5: 19,638 additional rows, total coverage 31.6% (from 21.7%)
  at confidence >= 0.7: 8,800 additional rows, total coverage 26.2% (from 21.7%)
  at confidence >= 0.9: 2,231 additional rows, total coverage 22.9% (from 21.7%)

Wrote model_predictions.csv
These still need checking against the hand-labeled gold set before we trust them.


## 13. Model comparison and feature ablation

Section 11 used one classifier. A single model does not tell us whether the approach is limited by the algorithm or by the features, so here we compare several on the same honest product split, then strip features away to see which ones carry the signal.

All numbers below are on the product split, so every test product is unseen during training.

In [20]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import SGDClassifier

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(lab, groups=lab['product']))
A, B = lab.iloc[tr], lab.iloc[te]
ytr, yte = y.iloc[tr], y.iloc[te]

v_c  = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=120000, min_df=2)
v_u  = TfidfVectorizer(analyzer='word')
Xtr = hstack([v_c.fit_transform(A['masked']), v_u.fit_transform(A[COL_UOM].fillna('').astype(str))]).tocsr()
Xte = hstack([v_c.transform(B['masked']),     v_u.transform(B[COL_UOM].fillna('').astype(str))]).tocsr()

print(f'majority-class baseline: {yte.value_counts(normalize=True).iloc[0]*100:.1f}%')
scores={}
for name, mdl in [('LinearSVC', LinearSVC()),
                  ('ComplementNB', ComplementNB()),
                  ('SGD modified-huber', SGDClassifier(loss='modified_huber', max_iter=30, random_state=0)),
                  ('LogisticRegression', LogisticRegression(max_iter=1000))]:
    mdl.fit(Xtr, ytr)
    a = accuracy_score(yte, mdl.predict(Xte))*100
    scores[name]=round(a,1)
    print(f'  {name:22s} {a:5.1f}%')
RESULTS['model_comparison']=scores

majority-class baseline: 11.8%
  LinearSVC               70.3%
  ComplementNB            48.2%
  SGD modified-huber      69.3%
  LogisticRegression      60.6%


In [21]:
# which features actually carry the signal
print('Feature ablation, LogisticRegression, product split:')
abl={}
mdl = LogisticRegression(max_iter=600).fit(Xtr, ytr)
abl['char 3-5 + UOM'] = round(accuracy_score(yte, mdl.predict(Xte))*100,1)

a2 = v_c.transform(A['masked']); b2 = v_c.transform(B['masked'])
mdl2 = LogisticRegression(max_iter=600).fit(a2, ytr)
abl['char 3-5 only'] = round(accuracy_score(yte, mdl2.predict(b2))*100,1)

wv = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=60000, min_df=2)
a3 = wv.fit_transform(A['masked']); b3 = wv.transform(B['masked'])
mdl3 = LogisticRegression(max_iter=600).fit(a3, ytr)
abl['word 1-2 only'] = round(accuracy_score(yte, mdl3.predict(b3))*100,1)

for k,vv in abl.items(): print(f'  {k:20s} {vv:5.1f}%')
RESULTS['feature_ablation']=abl

Feature ablation, LogisticRegression, product split:
  char 3-5 + UOM        60.6%
  char 3-5 only         62.2%
  word 1-2 only         60.8%


## 14. Error analysis

Knowing the accuracy is not the same as knowing what the model gets wrong. This looks at which pack sizes are confused for which, since that tells us whether errors are near misses or wild guesses, and whether particular container types are systematically harder.

In [22]:
best = LinearSVC().fit(Xtr, ytr)
pred = best.predict(Xte)
err = pd.DataFrame({'true':yte.values,'pred':pred,'uom':B[COL_UOM].values})
err['correct'] = err['true']==err['pred']

print('Accuracy by unit of measure (units with >=100 test rows):')
byu = err.groupby('uom').agg(n=('correct','size'), acc=('correct','mean'))
byu = byu[byu['n']>=100].sort_values('acc')
print((byu.assign(acc=lambda d:(d['acc']*100).round(1))).to_string())

print()
print('Most frequent confusions:')
conf = err[~err['correct']].groupby(['true','pred']).size().sort_values(ascending=False).head(10)
print(conf.to_string())

print()
mult = []
for t,p in zip(err.loc[~err['correct'],'true'], err.loc[~err['correct'],'pred']):
    try:
        tv,pv=float(t),float(p)
        if tv>0: mult.append(pv/tv)
    except ValueError: pass
if mult:
    mult=pd.Series(mult)
    print(f'Of the wrong predictions, {(mult.between(0.5,2)).mean()*100:.0f}% are within a factor of 2 of the truth.')
    print('Errors clustered near the true value are far less damaging to a price benchmark')
    print('than errors that are orders of magnitude off.')

Accuracy by unit of measure (units with >=100 test rows):
        n   acc
uom            
CR   1122  46.1
BX   3183  66.2
CS   3762  71.3
PK   3230  81.6

Most frequent confusions:
true    pred  
24.0    4.0       398
6.0     12.0      350
500.0   1000.0    207
2000.0  1000.0    189
250.0   20.0      112
200.0   100.0      80
32.0    80.0       53
1000.0  100.0      45
500.0   250.0      39
1000.0  500.0      38

Of the wrong predictions, 47% are within a factor of 2 of the truth.
Errors clustered near the true value are far less damaging to a price benchmark
than errors that are orders of magnitude off.


## 15. Item grouping: testing the bottleneck we found

Section 6 showed that the category field is too coarse to group comparable items, which is why normalizing inside those groups added noise instead of removing it. That was a diagnosis, not a fix. This section tests a fix.

The idea is to group items by how similar their descriptions are rather than by their assigned category. We build nearest-neighbour links between item descriptions, keep links above a similarity threshold, and treat each connected cluster as one product group. Then we measure the same thing we measured before: how much per-each price varies inside a group. Lower is better, because a group whose members have consistent prices is a group we can benchmark against and flag outliers within.

We compare three ways of grouping. The category and unit pairing is what the aggregation currently uses. Exact description matching is the ceiling, since identical text is certainly the same product, but it only works when text repeats exactly. Similarity grouping sits in between and is the one we are actually proposing.

In [23]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

grp = target.loc[resolved_mask].copy()
grp = grp[(grp['unit_price_each']>0) & (grp[COL_UPRICE]>0) & np.isfinite(grp['unit_price_each'])]
grp['norm'] = grp[COL_DESC].fillna('').astype(str).str.upper().str.strip()
print(f'rows available for grouping: {len(grp):,}')

def group_quality(labels, name, store=None):
    t = grp.copy(); t['g'] = labels
    sizes = t.groupby('g').size()
    keep = sizes[sizes>=5].index
    sub = t[t['g'].isin(keep)]
    cvs = sub.groupby('g')['unit_price_each'].apply(cv).dropna()
    med = cvs.median()
    print(f'  {name:32s} groups: {len(keep):6,d} | median size {sizes[keep].median():4.0f} | median within-group CV {med:.3f}')
    if store is not None: store[name]=round(float(med),3)
    return med

GQ={}
print('Grouping strategy comparison (groups of 5 or more rows):')
group_quality(grp[COL_CAT].astype(str)+'|'+grp[COL_UOM].astype(str), 'category + UOM (current)', GQ)
group_quality(grp['norm'], 'exact description (ceiling)', GQ)

rows available for grouping: 43,100
Grouping strategy comparison (groups of 5 or more rows):
  category + UOM (current)         groups:    532 | median size   17 | median within-group CV 0.897
  exact description (ceiling)      groups:  1,457 | median size    9 | median within-group CV 0.043


np.float64(0.04324204706930611)

In [24]:
vg = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=150000, min_df=2)
Xg = vg.fit_transform(grp['norm'])
K = 10
nn = NearestNeighbors(n_neighbors=K, metric='cosine').fit(Xg)
dist, idx = nn.kneighbors(Xg)
sim = 1 - dist
n = len(grp)

for thr in [0.5,0.6,0.7,0.8,0.9]:
    rows=[]; cols=[]
    for i in range(n):
        for j in range(1,K):
            if sim[i,j] >= thr:
                rows.append(i); cols.append(idx[i,j])
    if not rows: continue
    Adj = csr_matrix((np.ones(len(rows)),(rows,cols)), shape=(n,n))
    _, labels = connected_components(Adj, directed=False)
    group_quality(labels, f'similarity grouping thr={thr}', GQ)

RESULTS['grouping_quality']=GQ
print()
print('Lower within-group CV means the items inside a group really are comparable,')
print('which is the precondition for a price benchmark or an outlier flag to mean anything.')

  similarity grouping thr=0.5      groups:  1,181 | median size   14 | median within-group CV 0.195
  similarity grouping thr=0.6      groups:  1,349 | median size   12 | median within-group CV 0.155
  similarity grouping thr=0.7      groups:  1,465 | median size   11 | median within-group CV 0.124
  similarity grouping thr=0.8      groups:  1,506 | median size   10 | median within-group CV 0.096
  similarity grouping thr=0.9      groups:  1,523 | median size    9 | median within-group CV 0.073

Lower within-group CV means the items inside a group really are comparable,
which is the precondition for a price benchmark or an outlier flag to mean anything.


## 16. End to end demonstration

Putting the pieces together on the actual center question to this project. We take the similarity groups, use normalized per-each prices, and flag purchases that sit far from the group's typical price using a robust score that is not distorted by a few extreme values. We then compare what we would have flagged using raw unit prices and the category grouping, which is the status quo.

In [25]:
thr = 0.7
rows=[]; cols=[]
for i in range(n):
    for j in range(1,K):
        if sim[i,j] >= thr:
            rows.append(i); cols.append(idx[i,j])
Adj = csr_matrix((np.ones(len(rows)),(rows,cols)), shape=(n,n))
_, glabels = connected_components(Adj, directed=False)
grp['group'] = glabels

sizes = grp.groupby('group').size()
usable = sizes[sizes>=8].index
sub = grp[grp['group'].isin(usable)].copy()
print(f'Groups usable for benchmarking (8+ rows): {len(usable):,} covering {len(sub):,} line items')

def rz(s):
    med=s.median(); mad=(s-med).abs().median()
    if not np.isfinite(mad) or mad < 1e-6: return s*0
    return (s-med)/(1.4826*mad)

sub['z_new'] = sub.groupby('group')['unit_price_each'].transform(lambda s: rz(np.log(s)))
sub['z_old'] = sub.groupby([COL_CAT,COL_UOM])[COL_UPRICE].transform(lambda s: rz(np.log(s)))
# A robust score alone is not enough. In groups where nearly every price is identical the
# spread is tiny, so a price only slightly off the median can score in the thousands. We
# require a flag to be both statistically unusual and practically large, meaning at least
# 50 percent away from the group's typical per each price.
sub['group_median'] = sub.groupby('group')['unit_price_each'].transform('median')
sub['ratio'] = sub['unit_price_each'] / sub['group_median']
practical = (sub['ratio'] >= 1.5) | (sub['ratio'] <= 1/1.5)

sub['flag_stat_only'] = sub['z_new'].abs() > 3.5
sub['flag_new'] = sub['flag_stat_only'] & practical
sub['flag_old'] = sub['z_old'].abs() > 3.5
print(f"statistical test alone would flag : {int(sub['flag_stat_only'].sum()):,}")
print(f"after requiring a real price gap  : {int(sub['flag_new'].sum()):,}")

print(f"flagged by new approach : {sub['flag_new'].sum():,}")
print(f"flagged by status quo   : {sub['flag_old'].sum():,}")
print(f"agree on               : {(sub['flag_new']&sub['flag_old']).sum():,}")
print(f"only new flags         : {(sub['flag_new']&~sub['flag_old']).sum():,}")
print(f"only status quo flags  : {(~sub['flag_new']&sub['flag_old']).sum():,}")
RESULTS['flags_new']=int(sub['flag_new'].sum()); RESULTS['flags_old']=int(sub['flag_old'].sum())

print()
print('Highest-confidence flags under the new approach:')
top = sub[sub['flag_new']].reindex(sub[sub['flag_new']]['z_new'].abs().sort_values(ascending=False).index)
print(top[[COL_DESC,COL_UOM,COL_UPRICE,'pack_size','unit_price_each','ratio','z_new']].head(8).to_string(index=False))
print()
print('Treat these as a review queue, not findings. Some of the largest gaps trace back to')
print('our own conversion errors rather than to real overpayment, which is what the labeled')
print('gold set is meant to separate.')
sub.to_csv('anomaly_candidates.csv', index=False)
print()
print('Wrote anomaly_candidates.csv')

Groups usable for benchmarking (8+ rows): 1,005 covering 31,130 line items
statistical test alone would flag : 1,589
after requiring a real price gap  : 746
flagged by new approach : 746
flagged by status quo   : 1,037
agree on               : 73
only new flags         : 673
only status quo flags  : 964

Highest-confidence flags under the new approach:
                                     LineItem.Description LineItem.UOM  LineItem.UnitPrice  pack_size  unit_price_each    ratio        z_new
           SPECULA, VAGINAL DISP W/LED MED (24/BX 4BX/CS)           BX              160.70       24.0         6.695833 1.988554 14987.257176
             SPECULA, VAGINAL DISP W/LED SM(24/BX 4BX/CS)           CS              541.96       96.0         5.645417 1.676597 11266.834207
             SPECULA, VAGINAL DISP W/LED SM(24/BX 4BX/CS)           CS              541.96       96.0         5.645417 1.676597 11266.834207
             SPECULA, VAGINAL DISP W/LED SM(24/BX 4BX/CS)           BX           

## 17. Final summary

In [26]:
print('='*64)
print('FULL RESULTS')
print('='*64)
for k,val in RESULTS.items():
    if k=='top_examples': continue
    print(f'{k:28s} {val}')

FULL RESULTS
rows_loaded                  917321
price_identity_pct           100.0
uom_class_counts             {'each-like': 693107, 'multi-or-measure': 202643, 'service': 21571}
junk_rows                    38057
junk_pct                     4.15
working_rows                 879264
target_rows                  198337
ea_rows                      659617
resolved_rows                43130
resolved_pct                 21.7
method_counts                {'unresolved': 152332, 'direct-token': 38757, 'nested-token': 2284, 'inner-unit-only': 1912, 'unit-mismatch': 953, 'phrase-of': 940, 'per-token': 764, 'x-multiplier': 264, 'nested-direct-pair': 121, 'invalid-pack-size': 10}
declined_rows                2865
same_item_groups             58
same_item_improved           58
same_item_cv_raw             0.653
same_item_cv_norm            0.078
same_item_change_pct         -91.2
cat_groups                   190
cat_improved                 31
cat_change_pct               56.0
cat_median_items_p

## 18. Scoring against the hand-labeled gold set

Labelers write a number when the description makes the pack size determinable, and a
question mark when it does not. Those two answers mean different things and neither is
discarded.

A. Parser answered and the labeler determined the truth. These give precision.

B. Parser answered but the labeler could not tell. These cannot be scored. Dropping them
   silently would inflate precision, since they are exactly the hard cases, so we exclude
   them from the calculation and report how many there were.

C. Parser declined. Here a question mark is a good outcome. If a person reading the same
   description also cannot tell, the parser was right to decline. If the person could tell,
   the row was recoverable and represents headroom for the model.

In [27]:
LABELED = Path('gold_set_to_label.csv')
KEY     = Path('gold_set_ANSWERKEY.csv')
UNKNOWN_MARKS = {'?','??','unknown','n/a','na','cant tell','unclear','ambiguous','service'}

if not (LABELED.exists() and KEY.exists()):
    print('Gold set files not found. Run the export cell in section 9 first.')
else:
    lab_g = pd.read_csv(LABELED); key_g = pd.read_csv(KEY)
    raw = lab_g['true_pack_size'].astype(str).str.strip()
    lab_g['true']       = pd.to_numeric(raw, errors='coerce')
    lab_g['unknowable'] = raw.str.lower().isin(UNKNOWN_MARKS)
    lab_g['untouched']  = raw.str.lower().isin(['','nan','none']) & ~lab_g['unknowable']

    j = key_g.merge(lab_g[['row_id','true','unknowable','untouched']], on='row_id', how='inner')
    j['reviewed'] = ~j['untouched']; j['parser_answered'] = j['pack_size'].notna()

    print(f"gold rows             : {len(j)}")
    print(f"reviewed by a labeler : {int(j['reviewed'].sum())}")
    print(f"  gave a number       : {int(j['true'].notna().sum())}")
    print(f"  marked unknowable   : {int(j['unknowable'].sum())}")
    print(f"not yet labeled       : {int(j['untouched'].sum())}")

    if j['reviewed'].sum() == 0:
        print('\nNo labels yet, nothing to score.')
    else:
        R = j[j['reviewed']]
        A = R[R['parser_answered'] & R['true'].notna()].copy()
        print('\nA. PRECISION  (parser answered, truth known)')
        if len(A):
            A['correct'] = np.isclose(A['pack_size'], A['true'], rtol=0.01)
            print(f"   n = {len(A)}   precision = {A['correct'].mean()*100:.1f}%")
            RESULTS['gold_precision'] = round(float(A['correct'].mean()*100),1)
            RESULTS['gold_n_scored']  = int(len(A))
            for tier in ['high','medium']:
                t = A[A['confidence'] == tier]
                if len(t): print(f"     {tier:7s} n={len(t):3d}   {t['correct'].mean()*100:5.1f}%")

        B = R[R['parser_answered'] & R['unknowable']]
        n_ans = int(R['parser_answered'].sum())
        print('\nB. UNVERIFIABLE  (parser answered, labeler could not tell)')
        if n_ans:
            print(f"   n = {len(B)} of {n_ans} reviewed parser answers ({len(B)/n_ans*100:.0f}%)")
            RESULTS['gold_unverifiable_pct'] = round(len(B)/n_ans*100,1)

        C = R[~R['parser_answered']]
        print('\nC. DECLINE QUALITY  (parser gave no answer)')
        if len(C):
            good = int(C['unknowable'].sum()); rec = int(C['true'].notna().sum())
            print(f"   n = {len(C)}")
            print(f"   labeler also could not tell : {good} ({good/len(C)*100:.0f}%)  correct decline")
            print(f"   labeler could tell          : {rec} ({rec/len(C)*100:.0f}%)  recoverable headroom")
            RESULTS['gold_correct_declines_pct'] = round(good/len(C)*100,1)
            for b in C['bucket'].dropna().unique():
                s = C[C['bucket'] == b]
                print(f"     {b:22s} n={len(s):3d}  recoverable {int(s['true'].notna().sum())}")

        if len(A) and (~A['correct']).sum():
            print('\nD. DISAGREEMENTS between parser and labeler')
            print('   Review these by hand. Some will be parser errors and some will be')
            print('   labeling errors, and the distinction matters for the reported number.')
            print(A[~A['correct']][['row_id',COL_DESC,COL_UOM,'pack_size','true','method','evidence']].head(20).to_string(index=False))

gold rows             : 300
reviewed by a labeler : 300
  gave a number       : 252
  marked unknowable   : 46
not yet labeled       : 0

A. PRECISION  (parser answered, truth known)
   n = 136   precision = 98.5%
     high    n=102   100.0%
     medium  n= 34    94.1%

B. UNVERIFIABLE  (parser answered, labeler could not tell)
   n = 14 of 150 reviewed parser answers (9%)

C. DECLINE QUALITY  (parser gave no answer)
   n = 150
   labeler also could not tell : 32 (21%)  correct decline
   labeler could tell          : 116 (77%)  recoverable headroom
     no-token               n= 90  recoverable 63
     declined-inner-unit    n= 40  recoverable 36
     declined-mismatch      n= 20  recoverable 17

D. DISAGREEMENTS between parser and labeler
   Review these by hand. Some will be parser errors and some will be
   labeling errors, and the distinction matters for the reported number.
 row_id                         LineItem.Description LineItem.UOM  pack_size   true       method evidence
 

## 19. Item grouping as a reusable function

Section 15 swept similarity thresholds to show that grouping by description beats grouping by category. This turns that finding into something someone else can run. The function takes descriptions and returns a group id per row, using the threshold the sweep supported.

We settled on 0.7. Below it, groups start merging unrelated products and within-group price spread climbs. Above it, groups fragment toward exact string matching and we lose the ability to connect items that are the same product written two different ways. 0.7 keeps within-group variability close to the exact-match ceiling while still grouping variant spellings.

In [28]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

def group_items(descriptions, threshold=0.7, n_neighbors=10, min_df=2):
    """Assign a group id to each description.

    Items whose descriptions are similar above the threshold are linked, and each
    connected cluster becomes one group. Returns an array of group ids aligned to
    the input order.
    """
    texts = pd.Series(descriptions).fillna('').astype(str).str.upper().str.strip()
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=150000, min_df=min_df)
    X = vec.fit_transform(texts)
    k = min(n_neighbors, X.shape[0])
    nn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(X)
    dist, idx = nn.kneighbors(X)
    sim = 1 - dist
    rows=[]; cols=[]
    for i in range(X.shape[0]):
        for j in range(1, k):
            if sim[i, j] >= threshold:
                rows.append(i); cols.append(idx[i, j])
    if not rows:
        return np.arange(X.shape[0])
    adj = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(X.shape[0],)*2)
    _, labels = connected_components(adj, directed=False)
    return labels

# apply it and report what we get
priced = target.loc[resolved_mask].copy()
priced = priced[(priced['unit_price_each']>0) & np.isfinite(priced['unit_price_each'])]
priced['group_id'] = group_items(priced[COL_DESC])

sizes = priced.groupby('group_id').size()
print(f'rows grouped        : {len(priced):,}')
print(f'groups formed       : {priced["group_id"].nunique():,}')
print(f'groups with 5+ rows : {(sizes>=5).sum():,}')
cvs = priced[priced['group_id'].isin(sizes[sizes>=5].index)].groupby('group_id')['unit_price_each'].apply(cv).dropna()
print(f'median within-group price variability: {cvs.median():.3f}')
RESULTS['grouping_median_cv'] = round(float(cvs.median()),3)
RESULTS['grouping_n_groups']  = int(priced['group_id'].nunique())

rows grouped        : 43,100
groups formed       : 7,470
groups with 5+ rows : 1,465
median within-group price variability: 0.124


## 20. Handoff package

Writes the pipeline outputs plus a plain text README describing what the pipeline does, what it does not do, and the limitations a reader needs to know before using any of it.

In [29]:
priced.to_csv('normalized_line_items.csv', index=False)

readme = f'''GovSpend Quantity Normalization Pipeline
UVA MSDS Capstone, DS6015

WHAT THIS DOES
Converts public procurement purchase order line items to a per-each price so that
purchases priced by the case, box or pack can be compared against each other and
against items already priced individually.

HOW IT WORKS
1. Removes rows that are not products (freight, tax, order summaries, spreadsheet errors).
2. Sorts unit of measure codes into three groups: already per item, time based service
   codes that cannot be converted, and multi unit containers that need conversion.
3. Reads the pack quantity out of the item description and verifies that the unit it
   found matches the unit the row is priced in before accepting it. Where the two
   disagree the row is declined rather than guessed at.
4. Trains a classifier on the rows the rules resolved and uses it to predict pack sizes
   for rows the rules could not handle.
5. Groups comparable items by description similarity and flags purchases whose per each
   price is far from the typical price for their group.

RESULTS ON THE CURRENT EXTRACT
Line items processed              {RESULTS.get("rows_loaded","")}
Rows needing conversion           {RESULTS.get("target_rows","")}
Resolved by rules                 {RESULTS.get("resolved_rows","")} ({RESULTS.get("resolved_pct","")}%)
Declined rather than guessed      {RESULTS.get("declined_rows","")}
Model accuracy, unseen products   {RESULTS.get("ml_acc_product_split","")}%
Median within group price spread  {RESULTS.get("grouping_median_cv","")} using description grouping

KNOWN LIMITATIONS
1. Pack sizes are not yet verified against hand checked ground truth. Precision figures
   are pending the labeled evaluation set.
2. The supplied category field is unreliable at the line level and should not be used to
   group items for price comparison. This is why the pipeline groups on descriptions.
3. Rows coded as each are not always priced per each. Roughly {RESULTS.get("ea_suspect_rows","")} rows carry a
   pack count in the description despite an each unit code.
4. Outlier output at this stage surfaces conversion errors as well as genuine price
   differences. It should be treated as a review queue, not a findings list.
5. Coverage is partial. Most rows still lack any pack information in the description.

FILES
normalized_line_items.csv  line items with per each price and group id
resolved_baseline.csv      rows converted by the rules, with the evidence for each
unresolved_for_llm.csv     rows not converted, with the reason
model_predictions.csv      model predictions for unresolved rows above the confidence cut
gold_set_to_label.csv      evaluation sample for hand labeling
anomaly_candidates.csv     grouped items with price outlier flags

RERUNNING
Open the notebook, set CSV_PATH to the extract, and run all cells in order.
'''
with open('README.txt','w') as f:
    f.write(readme)
print(readme)

GovSpend Quantity Normalization Pipeline
UVA MSDS Capstone, DS6015

WHAT THIS DOES
Converts public procurement purchase order line items to a per-each price so that
purchases priced by the case, box or pack can be compared against each other and
against items already priced individually.

HOW IT WORKS
1. Removes rows that are not products (freight, tax, order summaries, spreadsheet errors).
2. Sorts unit of measure codes into three groups: already per item, time based service
   codes that cannot be converted, and multi unit containers that need conversion.
3. Reads the pack quantity out of the item description and verifies that the unit it
   found matches the unit the row is priced in before accepting it. Where the two
   disagree the row is declined rather than guessed at.
4. Trains a classifier on the rows the rules resolved and uses it to predict pack sizes
   for rows the rules could not handle.
5. Groups comparable items by description similarity and flags purchases whose per ea